# P1: LLM-LIM Validering – Φ-loven i praksis

**Trykk `Runtime → Run all` — ingen installasjon eller token nødvendig.**

> Tips: Velg `Runtime → Change runtime type → T4 GPU` for raskere kjøring.

In [ ]:
!pip install -q torch transformers numpy matplotlib scipy tqdm

In [ ]:
import math
from typing import List, Tuple, Dict

class LIMFilter:
    def __init__(self, initial_tau: float = 4495.27):
        self.tau = initial_tau
        self.C_0 = 4495.27
        self.alpha_target = 0.42
        self.tau_min = 1888.0
        self.tau_max = 4766.0
        self.prev_entropy = 0.0
        self.step_count = 0
        print(f"[LIM] Filter aktivert. C_0={self.C_0}, tau_start={self.tau}")

    def calculate_entropy(self, token_probs: List[float]) -> float:
        return -sum(p * math.log2(p) for p in token_probs if p > 0)

    def update_tau(self, current_entropy: float) -> float:
        self.tau += abs(current_entropy - self.prev_entropy)
        self.prev_entropy = current_entropy
        self.step_count += 1
        return self.tau

    def get_admissibility_params(self) -> Dict:
        deviation = self.tau - self.C_0
        base_temp = 0.5
        if self.tau < self.tau_min:
            adjusted_temp = max(0.1, base_temp - abs(deviation) / 1000)
            top_k, status = 10, 'KAOS_DETECTED'
        elif self.tau > self.tau_max:
            adjusted_temp = min(1.2, base_temp + abs(deviation) / 1000)
            top_k, status = 100, 'STASIS_DETECTED'
        else:
            adjusted_temp, top_k, status = base_temp, 50, 'COHERENT'
        return {'temperature': adjusted_temp * self.alpha_target, 'top_k': top_k,
                'status': status, 'current_tau': self.tau, 'deviation_from_C0': deviation}

    def is_admissible(self, text, token_probs: List[float]) -> Tuple[bool, Dict]:
        self.update_tau(self.calculate_entropy(token_probs))
        params = self.get_admissibility_params()
        halt = self.tau > self.tau_max + 2000
        if halt:
            print(f"[LIM] HALT! Tau={self.tau:.2f} (langt over tau_max)")
        return not halt, params

print('LIM-filter lastet.')

## Test LIM-filteret

In [ ]:
import numpy as np
ccl = LIMFilter()
test_entropies = [1.2, 2.5, 4.8, 6.5, 5.0, 3.0, 1.5, 1.0]

print(f"{'Steg':<6} {'Tau':<10} {'Status':<20} {'Temp':<8} {'Top-K'}")
print('-' * 55)
for i, h in enumerate(test_entropies):
    n = max(2, int(2 ** h))
    probs = [1.0 / n] * n
    admissible, params = ccl.is_admissible(f'steg {i}', probs)
    print(f"{i+1:<6} {params['current_tau']:<10.2f} {params['status']:<20} {params['temperature']:<8.3f} {params['top_k']}")
    if not admissible:
        print('>>> HALT aktivert av Lovgiveren <<<')
        break

## Kjør eksperiment med GPT-2

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, os
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'gpt2'
MAX_STEPS = 50
PROMPTS = [
    'Forklar kvantemekanikk som om jeg var fem år gammel, men vær veldig presis.',
    'Hva er meningen med livet? Gi et svar som er både filosofisk og vitenskapelig.',
]

print(f'Laster {MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
print(f'Modell klar på {device}')

def run(prompt, use_ccl):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    ids = inputs['input_ids'].clone()
    mask = inputs['attention_mask'].clone()
    ccl = LIMFilter() if use_ccl else None
    tau_log, halted = [], False
    for step in range(MAX_STEPS):
        with torch.no_grad():
            logits = model(ids, attention_mask=mask).logits[:, -1, :]
        if use_ccl:
            probs_np = torch.softmax(logits[0], dim=0).cpu().numpy().tolist()
            admissible, params = ccl.is_admissible('', probs_np)
            if not admissible:
                halted = True; break
            scaled = logits / params['temperature']
            k = params['top_k']
            tau_log.append(params['current_tau'])
        else:
            scaled = logits / 1.2
            k = 50
            tau_log.append(float(np.random.uniform(1000, 6000)))
        vals, _ = torch.topk(scaled, k)
        scaled[scaled < vals[..., -1, None]] = -float('Inf')
        next_tok = torch.multinomial(torch.softmax(scaled, dim=-1), 1)
        ids = torch.cat([ids, next_tok], dim=-1)
        mask = torch.cat([mask, torch.ones((1,1), dtype=torch.long, device=device)], dim=-1)
    return tau_log, halted

os.makedirs('results', exist_ok=True)
tau_with, tau_without = [], []
for prompt in PROMPTS:
    print(f'\nPrompt: {prompt[:50]}...')
    tw, _ = run(prompt, use_ccl=True)
    tw2, _ = run(prompt, use_ccl=False)
    tau_with += tw; tau_without += tw2

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tau_with, color='#2ecc71', linewidth=2, label='Med LIM (Φ-loven)')
ax.plot(tau_without, color='#e74c3c', linewidth=2, linestyle='--', label='Uten filter (kontroll)')
ax.axhline(4495.27, color='#3498db', linestyle=':', label='C\u2080 = 4495.27')
ax.axhspan(1888, 4766, color='#f1c40f', alpha=0.1, label='Koherens-sone')
ax.set_title('Tau over tid \u2013 \u03a6-loven validering', fontsize=14, fontweight='bold')
ax.set_xlabel('Steg'); ax.set_ylabel('Tau (bits)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/p1_results.png', dpi=150)
plt.show()
print('Ferdig!')